# Video Deepfake Detection — Colab Pipeline

**Train on FaceForensics++ (c23). Test on Celeb-DF v2, FF++ c40, and DFD.**

This is the standard academic protocol ([DeepfakeBench](https://github.com/SCLBD/DeepfakeBench)
trains on FF++ c23 and cross-tests elsewhere), so your numbers are directly
comparable to published work — and all of it fits on Colab.

What you get out of it:

| Number | Where it comes from |
|---|---|
| In-domain performance | FF++ c23 held-out test split |
| **Codec robustness** | The same test videos at **c40** — real H.264 compression, free |
| **Cross-dataset drop** | **Celeb-DF v2** — the hardest standard cross-test |
| Second cross-dataset point | **DFD** — ships with the same FF++ downloader |
| Unseen-generator drop | Optional: hold out one FF++ method (e.g. NeuralTextures) |

**Before you start you need two things:**

1. The FF++ downloader script. Fill the form at
   [github.com/ondyari/FaceForensics](https://github.com/ondyari/FaceForensics)
   and they email you `faceforensics_download_v4.py`. Put it next to the
   project files.
2. Celeb-DF v2 access — the form at
   [github.com/yuezunli/celeb-deepfakeforensics](https://github.com/yuezunli/celeb-deepfakeforensics),
   or a Kaggle mirror.

Run the cells top to bottom. Logic lives in the `.py` files, so any stage can
be rerun on its own after a disconnect.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 2. Mount Drive and put the project on the path

Checkpoints go to Drive so a disconnect costs one epoch, not the run. Videos
and crops stay on local session disk — reading thousands of small files through
the Drive FUSE layer can triple epoch time.

Upload the project `.py` files **and the FF++ downloader script** to
`DRIVE_PROJECT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys
from pathlib import Path

DRIVE_PROJECT = '/content/drive/MyDrive/ctf'   # where you uploaded the files
PROJECT_DIR   = '/content/ctf'                 # working copy on session disk

Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
if Path(DRIVE_PROJECT).exists():
    for pat in ('*.py', 'requirements.txt'):
        for f in Path(DRIVE_PROJECT).glob(pat):
            shutil.copy2(f, PROJECT_DIR)
    print('synced project from Drive')
else:
    print(f'NOTE: {DRIVE_PROJECT} not found — upload the .py files there.')

os.environ['DFD_DATA_ROOT']  = '/content/data'
os.environ['DFD_CKPT_DIR']   = '/content/drive/MyDrive/dfd_ckpts'
os.environ['DFD_REPORT_DIR'] = '/content/drive/MyDrive/dfd_reports'

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
import config; config.ensure_dirs()
print('data   ', config.DATA_ROOT)
print('ckpts  ', config.CKPT_DIR)
print('reports', config.REPORT_DIR)

# The FF++ downloader must be here too.
import glob
print('FF++ downloader present:', bool(glob.glob('faceforensics_download*.py')))

## 3. Install dependencies

`facenet-pytorch` is installed with `--no-deps` on purpose. Its dependency
block pins an older torch, and without the flag pip silently downgrades
Colab's torch and breaks CUDA.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-deps facenet-pytorch==2.6.0

import torch, torchvision
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
from facenet_pytorch import MTCNN
print("facenet-pytorch OK")

## 4. Smoke test — run before downloading anything

Thirty seconds, no data required. It checks the FF++ group key actually merges
each fake with the originals it was built from, that Celeb-DF labels are taken
from directories rather than a list file whose convention could be read
backwards, that the split plan is a real partition, plus transforms,
calibration, aggregation, video decoding and a full analyzer pass.

If this fails, fix it before spending an hour on downloads.

In [ ]:
!python smoke_test.py

## 5. Download FaceForensics++ (c23) — the training set

`download_data.py` feeds the official downloader's terms-of-use prompt a
newline so it doesn't hang in a notebook. **You are accepting the FF++ terms**
— the URL is printed when it runs; read it.

`--server EU2` is required: the FaceForensics authors have confirmed EU2 is
currently the only server running. The other two will just fail to connect.

Start with `--num-videos 200` for a first pass (fast, ~20 min). Drop the flag
for the full 1000 once you know the pipeline works end to end.

In [ ]:
# All five manipulations + originals, at c23.
!python download_data.py ffpp \
    --script faceforensics_download_v4.py \
    --root /content/data/ffpp \
    --compression c23 \
    --server EU2 --num-videos 200

!du -sh /content/data/ffpp 2>/dev/null | tail -1

## 6. Download the test sets

Three of them, and none costs much:

- **FF++ c40** — the same videos, heavily compressed. This is your codec
  robustness test, and it is *real* H.264 re-encoding rather than a simulated
  proxy.
- **DFD** — Google/Jigsaw actors, from the same downloader. A free second
  cross-dataset point.
- **Celeb-DF v2** — the hardest standard cross-dataset test.

In [ ]:
# FF++ c40 — same videos, heavier compression (robustness test set)
!python download_data.py ffpp \
    --script faceforensics_download_v4.py \
    --root /content/data/ffpp --compression c40 --server EU2 --num-videos 200

In [ ]:
# DFD (Google/Jigsaw actors) — free second cross-dataset point
!python download_data.py dfd \
    --script faceforensics_download_v4.py \
    --root /content/data/ffpp --compression c23 --server EU2

**Note on the Celeb-DF link:** opening it shows what looks like a folder, but
it is actually Drive's built-in preview of a single `.zip` file. There is
nothing to "open" into a folder of separate videos — the whole dataset is one
archive.

FIRST, outside Colab:
1. Open the Drive link the authors emailed you.
2. Right-click the zip file's name (not a folder) → **Organise** → **Add
   shortcut to Drive** → **My Drive**.

"Add shortcut to Drive" works on a file exactly as it does on a folder, so this
makes the zip visible at a path under `/content/drive/MyDrive/` once mounted.
`download_data.py` then copies that zip to local session disk and extracts it
there — extracting a multi-GB archive straight off the Drive mount is slow, a
local copy first is not.

In [ ]:
!ls /content/drive/MyDrive | head -30

In [ ]:
# Set this to the file (or folder) name you saw above -- it may differ
# slightly, e.g. "Celeb-DF-v2.zip" or "Celeb-DF (v2).zip".
CELEB_PATH = '/content/drive/MyDrive/Celeb-DF-v2.zip'

import subprocess, sys
subprocess.run([sys.executable, 'download_data.py', 'celebdf',
                '--root', '/content/data/celebdf',
                '--via', 'drive', '--drive-dir', CELEB_PATH])

## 7. Plan the splits — once, over groups, written down

This is where leakage is prevented. Every FF++ group is assigned to exactly one
of train / val / test / **clip**, and the assignment is saved so every later
script agrees and the decision is auditable.

The `clip` split is whole videos reserved for clip-level calibration.
`extract_crops.py` refuses to crop them, so the calibrator is fitted on video
the model has genuinely never seen.

Add `--holdout-method NeuralTextures` to also reserve one manipulation as an
unseen-generator test.

In [ ]:
!python plan_splits.py \
    --dataset ffpp --root /content/data/ffpp --compression c23 \
    --train 0.60 --val 0.15 --test 0.15 --clip 0.10 \
    --out /content/data/splits.json

## 8. Extract face crops

Crops are produced by the same function inference calls, so training crops and
inference crops are identical by construction — no crop-geometry mismatch is
possible.

FF++ pairs each real video with one fake per method, so reals are outnumbered
5:1. `--real-frames-per-video` defaults to 5× the fake rate, which rebalances
without discarding any fakes.

In [ ]:
# Training crops (train/val/test splits; the clip split is skipped automatically)
!python extract_crops.py \
    --dataset ffpp --root /content/data/ffpp --compression c23 \
    --splits /content/data/splits.json \
    --frames-per-video 12 \
    --out-dir /content/data/crops/ffpp_c23 \
    --out-manifest /content/data/ffpp_c23_manifest.csv

In [ ]:
# c40 crops, test split only — the codec robustness set
!python extract_crops.py \
    --dataset ffpp --root /content/data/ffpp --compression c40 \
    --splits /content/data/splits.json --only-split test \
    --frames-per-video 12 \
    --out-dir /content/data/crops/ffpp_c40 \
    --out-manifest /content/data/ffpp_c40_manifest.csv

# Celeb-DF, official 518-video test list only
!python extract_crops.py \
    --dataset celebdf --root /content/data/celebdf --official-test-only \
    --frames-per-video 12 \
    --out-dir /content/data/crops/celebdf \
    --out-manifest /content/data/celebdf_manifest.csv

# DFD
!python extract_crops.py \
    --dataset dfd --root /content/data/ffpp --compression c23 \
    --frames-per-video 8 \
    --out-dir /content/data/crops/dfd \
    --out-manifest /content/data/dfd_manifest.csv

## 9. One-epoch training check

Step 4 proved the plumbing; this proves the training loop runs on *your* data
in about a minute, before you commit to a full run.

In [ ]:
!python train.py --manifest /content/data/ffpp_c23_manifest.csv \
    --epochs 1 --limit 300 --batch-size 32 --ckpt-dir /content/smoke_ckpts

## 10. Train

Checkpoints every epoch to Drive. Model selection is on validation ROC-AUC, not
training loss — with near-duplicate crops the training loss falls steadily
while the model memorises scenes.

Watch the leakage tripwire: epoch-1 validation AUC above 0.98 means the split is
almost certainly leaking. On a correct group split that does not happen.

In [ ]:
!python train.py --manifest /content/data/ffpp_c23_manifest.csv \
    --arch resnet18 --epochs 10 --batch-size 64 --lr 1e-4 --patience 3

## 11. Evaluate — in-domain, per-method, and image-level robustness

`--by-method` tells you which manipulation is hardest, which is a good line for
the writeup. Accuracy is always printed next to the majority-class baseline.

In [ ]:
!python evaluate.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --manifest /content/data/ffpp_c23_manifest.csv \
    --by-method --robustness

## 12. The three generalization tests

Each of these is a separate, honestly-labelled number. A substantial drop is the
expected result — reporting it is the point of the exercise.

In [ ]:
# a) Codec robustness: same videos, c40 instead of c23. REAL compression.
!python evaluate.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --manifest /content/data/ffpp_c23_manifest.csv \
    --cross-manifest /content/data/ffpp_c40_manifest.csv

In [ ]:
# b) Cross-dataset: Celeb-DF v2, the hardest standard test.
!python evaluate.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --manifest /content/data/ffpp_c23_manifest.csv \
    --cross-manifest /content/data/celebdf_manifest.csv

In [ ]:
# c) Cross-dataset: DFD.
!python evaluate.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --manifest /content/data/ffpp_c23_manifest.csv \
    --cross-manifest /content/data/dfd_manifest.csv

## 13. Clip-level calibration and thresholds

This is what makes the confidence number honest. Frame-level temperature
scaling calibrates individual crops; it does not transfer to a clip aggregate,
because "22 of 30 frames flagged" is a count, not a probability.

This runs the full video pipeline over the reserved `clip` split, fits a
calibrator on aggregate features, tunes both thresholds against an explicit
error cost, and draws a **clip-level** reliability diagram — the level the
interface actually displays.

`--fp-cost 3` says wrongly calling an authentic official video synthetic is
three times worse than missing a fake. Change it if your writeup argues
differently, and state the value you used.

In [ ]:
!python fit_clip_calibration.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --dataset ffpp --root /content/data/ffpp --compression c23 \
    --splits /content/data/splits.json --split-name clip \
    --fp-cost 3 \
    --video-robustness

## 14. Launch the interface

`--share` gives a public link that expires after 72 hours. If the demo is being
judged later, relaunch it on the day.

In [ ]:
!python app.py \
    --checkpoint /content/drive/MyDrive/dfd_ckpts/model_best.pt \
    --share

## 15. Before you write it up

Numbers you should have, and the honest framing for each:

- [ ] **In-domain FF++ c23** — accuracy *next to* the majority baseline,
      precision/recall for both classes, ROC-AUC, PR-AUC, confusion matrix.
- [ ] **Per-method breakdown** — which manipulation is hardest. NeuralTextures
      usually is.
- [ ] **c23 → c40 drop** — real codec compression, not a simulated proxy. Say so
      explicitly, and keep it separate from the image-level sweep.
- [ ] **Cross-dataset drop on Celeb-DF** — expected to be large. This is the
      headline honesty number.
- [ ] **Cross-dataset drop on DFD** — a second, independent point.
- [ ] **Clip-level holdout metrics** and the clip reliability diagram, with the
      clip count stated. Ten bins over 60 clips is mostly noise; the script
      drops to 5 bins and says so.
- [ ] **The group key.** Say that FF++ source/target pairs were merged with
      union-find so a fake and its original can never straddle the split.
- [ ] **Effective sample size** — number of source groups, not crops.
- [ ] **Prior shift** — FF++ as extracted here is roughly balanced, but the
      deployment base rate for government communications is overwhelmingly
      authentic. Say whether you applied `calibrate.adjust_prior` or are
      reporting the caveat.
- [ ] **Domain gap** — FF++ is YouTube talking-heads; Celeb-DF is celebrity
      interview footage; the target is officials in broadcast conditions.

Three things worth getting right in the text:

- **Academic benchmarks overstate real-world performance.** Detectors that
  score well on FF++ and Celeb-DF still fail on deepfakes actually circulating
  online ([Fit for Purpose?, 2025](https://arxiv.org/pdf/2510.16556)). Naming
  this is stronger than hoping nobody asks.
- **FF++ fakes are mostly 2019-era methods.** Current generators are better.
  [DF40](https://github.com/YZY-stack/DF40) covers 40 modern techniques if you
  can get even a slice of it.
- **Grad-CAM shows where the model attended**, not evidence of manipulation. On
  a 7×7 feature map each cell covers roughly 32×32 input pixels.